In [43]:
# 판다스 넘파이 가져오기
import pandas as pd
import numpy as np

In [44]:
# 파일 불러오기
order = pd.read_csv('./data/orders.csv')
oi = pd.read_csv('./data/order_items.csv')
prod = pd.read_csv('./data/products.csv')
cust = pd.read_csv('./data/customers.csv')

In [45]:
# orders 파일 정제
print(f'주문일시 결측 주문 건수 : {order["order_datetime"].isna().sum()}')
orders=order.dropna(subset='order_datetime')
print(f'주문일시 결측 제거 후 주문 건수 : {orders.shape[0]}')
orders=orders.drop_duplicates(subset='order_id')
print(f'중복 제거 후 주문건 수 : {orders.shape[0]}')
orders['order_datetime']=pd.to_datetime(orders['order_datetime'],errors="coerce")
orders=orders[orders['order_datetime']<'2024-07-01']
orders=orders.loc[
    ~orders['status'].isin(['canceled','returned'])
].copy()
orders.info()

주문일시 결측 주문 건수 : 1933
주문일시 결측 제거 후 주문 건수 : 198067
중복 제거 후 주문건 수 : 198067
<class 'pandas.DataFrame'>
Index: 168312 entries, 0 to 199998
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        168312 non-null  int64         
 1   customer_id     168312 non-null  int64         
 2   order_datetime  168312 non-null  datetime64[us]
 3   channel         168312 non-null  str           
 4   status          168312 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 9.5 MB


In [46]:
orders['order_month']= orders['order_datetime'].dt.to_period('M')
orders['first_order']= (orders.groupby('customer_id')["order_month"].transform('min'))

orders['period']=(orders['order_month']-orders['first_order']).apply(lambda x: x.n)
orders.head()


,order_id,customer_id,order_datetime,channel,status,order_month,first_order,period
0,25463,1077,2024-06-25 00:17:41,store,delivered,2024-06,2024-01,5
1,171088,2998,2024-05-28 19:35:20,web,delivered,2024-05,2024-01,4
2,27437,4066,2024-02-14 17:49:14,app,delivered,2024-02,2024-02,0
3,98425,3243,2024-05-08 16:37:36,store,delivered,2024-05,2024-01,4
4,22325,5331,2024-04-17 20:08:22,app,delivered,2024-04,2024-01,3


In [50]:
cohort = orders.pivot_table(
    index='first_order',
    columns='period',
    values='customer_id',
    aggfunc='nunique'
)
cohort

period,0,1,2,3,4,5
first_order,,,,,,
2024-01,4165.0,3464.0,3687.0,3710.0,3767.0,3850.0
2024-02,786.0,568.0,598.0,596.0,619.0,NaN
2024-03,307.0,135.0,135.0,148.0,NaN,NaN
2024-04,184.0,31.0,34.0,NaN,NaN,NaN
2024-05,169.0,1.0,NaN,NaN,NaN,NaN
2024-06,182.0,NaN,NaN,NaN,NaN,NaN


In [53]:
cohort_size = cohort[0]
cohort_size

first_order
2024-01    4165.0
2024-02     786.0
2024-03     307.0
2024-04     184.0
2024-05     169.0
2024-06     182.0
Freq: M, Name: 0, dtype: float64

In [54]:
retention = cohort.div(cohort_size, axis=0)
retention

period,0,1,2,3,4,5
first_order,,,,,,
2024-01,1.0,0.831693,0.885234,0.890756,0.904442,0.92437
2024-02,1.0,0.722646,0.760814,0.758270,0.787532,NaN
2024-03,1.0,0.439739,0.439739,0.482085,NaN,NaN
2024-04,1.0,0.168478,0.184783,NaN,NaN,NaN
2024-05,1.0,0.005917,NaN,NaN,NaN,NaN
2024-06,1.0,NaN,NaN,NaN,NaN,NaN
